In [0]:
# ============================================================
# NOTEBOOK: nb_02_CustomerMasterExceptions
# PURPOSE:  Replaces SQL Procedure 2 (usp_LoadCustomerMasterExceptions)
#           Reads data quality issues, classifies them by severity,
#           assigns business owners, calculates SLAs, and MERGEs
#           into warehouse.customer_master_exceptions.
# CATALOG:  retailbank_dev
# ============================================================

import uuid
from datetime import datetime
from delta.tables import DeltaTable
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# --------------------------------------------------------
# PARAMETERS
# --------------------------------------------------------
dbutils.widgets.text("business_date", "2026-01-31", "Business Date")
dbutils.widgets.text("debug", "1", "Debug Mode (1=print, 0=silent)")

business_date = dbutils.widgets.get("business_date")
debug         = int(dbutils.widgets.get("debug"))

# --------------------------------------------------------
# AUDIT VARIABLES
# --------------------------------------------------------
execution_id   = str(uuid.uuid4())
procedure_name = "nb_02_CustomerMasterExceptions"
start_time     = datetime.now()

rows_read                   = 0
rows_inserted               = 0
rows_updated                = 0
duplicate_exceptions_removed = 0

if debug:
    print("=" * 50)
    print("NB_02_CUSTOMERMASTEREXCEPTIONS STARTED")
    print("=" * 50)
    print(f"Execution ID : {execution_id}")
    print(f"Business Date: {business_date}")

NB_02_CUSTOMERMASTEREXCEPTIONS STARTED
Execution ID : 1fd5a4c9-9dde-4f91-9c26-d3f924a490de
Business Date: 2026-01-31


In [0]:
# ============================================================
# AUDIT LOG FUNCTIONS
# We use spark.sql() to avoid DataFrame type-inference issues.
# ============================================================

def write_audit_start():
    sql = f"""
        INSERT INTO retailbank_dev.audit.etl_execution_log 
        (execution_id, procedure_name, business_date, start_time, status)
        VALUES 
        ('{execution_id}', '{procedure_name}', '{business_date}', 
         '{start_time.strftime("%Y-%m-%d %H:%M:%S")}', 'RUNNING')
    """
    spark.sql(sql)


def write_audit_end(status, message):
    end_time         = datetime.now()
    duration_seconds = int((end_time - start_time).total_seconds())
    safe_message     = message.replace("'", "''")
    
    sql = f"""
        UPDATE retailbank_dev.audit.etl_execution_log
        SET 
            end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
            status           = '{status}',
            rows_read        = {rows_read},
            rows_inserted    = {rows_inserted},
            rows_updated     = {rows_updated},
            rows_rejected    = {duplicate_exceptions_removed},
            duration_seconds = {duration_seconds},
            message          = '{safe_message}'
        WHERE execution_id = '{execution_id}'
    """
    spark.sql(sql)


write_audit_start()

In [0]:
# ============================================================
# READ DATA QUALITY ISSUES
# These were written by nb_01 (Procedure 1) when it rejected
# rows with NULL customer_ids.
# ============================================================

dq_df = spark.table("retailbank_dev.audit.data_quality_issues") \
    .filter(F.col("business_date") == business_date)

rows_read = dq_df.count()

if debug:
    print(f"Data Quality Records Loaded: {rows_read}")
    dq_df.show(truncate=False)

Data Quality Records Loaded: 11
+--------+-------------+-------------+-----------+-------------------+------------------------------------------+--------------------------+------------------------------------+
|issue_id|business_date|source_system|customer_id|error_category     |error_description                         |logged_date               |execution_id                        |
+--------+-------------+-------------+-----------+-------------------+------------------------------------------+--------------------------+------------------------------------+
|1       |2026-01-31   |CORE_BANKING |NULL       |MISSING_CUSTOMER_ID|Customer identifier is missing or invalid.|2026-08-16 22:53:54.798385|a81df450-e9dc-411a-b75d-952d62e549af|
|5       |2026-01-31   |CORE_BANKING |NULL       |MISSING_CUSTOMER_ID|Customer identifier is missing or invalid.|2026-08-23 03:52:52.014895|90dbc78a-1663-47d4-bc06-c98186292ce3|
|11      |2026-01-31   |CORE_BANKING |NULL       |MISSING_CUSTOMER_ID|Customer

In [0]:
# ============================================================
# DEDUPLICATION
# If the same exception is detected multiple times, keep only
# the latest one (by logged_date).
# 
# ROW_NUMBER() in PySpark uses a Window specification.
# This is the exact same logic as SQL Server's ROW_NUMBER().
# ============================================================

window_spec = Window.partitionBy(
    "customer_id", "error_category", "error_description"
).orderBy(F.desc("logged_date"))

deduped_df = dq_df.withColumn("rn", F.row_number().over(window_spec)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

duplicate_exceptions_removed = rows_read - deduped_df.count()

if debug:
    print(f"Duplicate Exceptions Removed : {duplicate_exceptions_removed}")
    print(f"Exceptions after dedup       : {deduped_df.count()}")

Duplicate Exceptions Removed : 10
Exceptions after dedup       : 1


In [0]:
# ============================================================
# READ CONFIGURATION TABLES
# These tell us how to classify each exception:
#   - ExceptionCategory  -> maps error type to severity + business area
#   - ExceptionSeverity  -> maps severity to SLA hours + escalation flag
#   - BusinessOwner      -> maps business area to named owner + support team
# ============================================================

cat_df = spark.table("retailbank_dev.config.exception_category")
sev_df = spark.table("retailbank_dev.config.exception_severity")
own_df = spark.table("retailbank_dev.config.business_owner")

if debug:
    print("Configuration loaded:")
    print(f"  Categories : {cat_df.count()}")
    print(f"  Severities : {sev_df.count()}")
    print(f"  Owners     : {own_df.count()}")

Configuration loaded:
  Categories : 6
  Severities : 4
  Owners     : 6


In [0]:
# ============================================================
# EXCEPTION CLASSIFICATION
# Two groups:
#   1. CLASSIFIED    -> found in config.exception_category
#   2. UNCLASSIFIED  -> not found; default to MEDIUM/48h/Data Governance
#
# We then UNION both groups together.
# ============================================================

# --- CLASSIFIED EXCEPTIONS ---
classified_df = deduped_df.alias("dq") \
    .join(cat_df.alias("cat"), F.col("dq.error_category") == F.col("cat.exception_category"), "inner") \
    .join(sev_df.alias("sev"), F.col("cat.default_severity") == F.col("sev.severity_code"), "inner") \
    .join(own_df.alias("own"), F.col("cat.business_area") == F.col("own.business_area"), "left") \
    .select(
        F.col("dq.customer_id"),
        F.col("dq.source_system").alias("source_system_code"),
        F.col("dq.error_category").alias("exception_category"),
        F.col("dq.error_description").alias("exception_description"),
        F.col("sev.severity_code"),
        F.concat(F.lit("Severity: "), F.col("sev.severity_code")).alias("severity_description"),
        F.col("sev.escalation_required"),
        F.col("sev.sla_hours").alias("resolution_target_hours"),
        F.col("cat.business_area"),
        F.col("own.business_owner"),
        F.col("own.support_team"),
        F.col("dq.logged_date"),
        F.lit(business_date).cast("date").alias("business_date"),
        F.lit("OPEN").alias("exception_status"),
        F.current_timestamp().alias("assigned_date"),
        F.lit("PENDING").alias("resolution_status")
    )

# --- UNCLASSIFIED EXCEPTIONS ---
# LEFT ANTI JOIN: keep only rows that do NOT exist in config
unclassified_df = deduped_df.alias("dq") \
    .join(cat_df.alias("cat"), F.col("dq.error_category") == F.col("cat.exception_category"), "left_anti") \
    .select(
        F.col("dq.customer_id"),
        F.col("dq.source_system").alias("source_system_code"),
        F.col("dq.error_category").alias("exception_category"),
        F.col("dq.error_description").alias("exception_description"),
        F.lit("MEDIUM").alias("severity_code"),
        F.lit("Medium Priority").alias("severity_description"),
        F.lit(False).alias("escalation_required"),
        F.lit(48).alias("resolution_target_hours"),
        F.lit("Data Governance").alias("business_area"),
        F.lit("Data Steward").alias("business_owner"),
        F.lit("Data Quality Team").alias("support_team"),
        F.col("dq.logged_date"),
        F.lit(business_date).cast("date").alias("business_date"),
        F.lit("OPEN").alias("exception_status"),
        F.current_timestamp().alias("assigned_date"),
        F.lit("PENDING").alias("resolution_status")
    )

# Union both groups
exceptions_df = classified_df.unionByName(unclassified_df, allowMissingColumns=True)

if debug:
    print("Exception Classification Summary:")
    exceptions_df.groupBy("severity_code").count().orderBy("severity_code").show()

Exception Classification Summary:
+-------------+-----+
|severity_code|count|
+-------------+-----+
|       MEDIUM|    1|
+-------------+-----+



In [0]:
# ============================================================
# ENRICH EXCEPTIONS
#   - EscalationLevel: CRITICAL=1, HIGH=2, MEDIUM=3, LOW=4
#   - ResolutionDueDate: LoggedDate + SLAHours
#   - Prefix [CRITICAL] to description for critical items
# ============================================================

exceptions_df = exceptions_df \
    .withColumn("escalation_level", 
        F.when(F.col("severity_code") == "CRITICAL", 1)
         .when(F.col("severity_code") == "HIGH",      2)
         .when(F.col("severity_code") == "MEDIUM",    3)
         .otherwise(4)
    ) \
    .withColumn("resolution_due_date",
        # Convert logged_date to seconds, add hours*3600, convert back
        F.from_unixtime(F.unix_timestamp("logged_date") + F.col("resolution_target_hours") * 3600)
         .cast("timestamp")
    ) \
    .withColumn("exception_description",
        F.when(F.col("severity_code") == "CRITICAL",
               F.concat(F.lit("[CRITICAL] "), F.col("exception_description")))
         .otherwise(F.col("exception_description"))
    )

if debug:
    print("SLA and Escalation calculated:")
    exceptions_df.select(
        "customer_id", "severity_code", "escalation_level", 
        "resolution_target_hours", "resolution_due_date"
    ).show(5, truncate=False)

SLA and Escalation calculated:
+-----------+-------------+----------------+-----------------------+-------------------+
|customer_id|severity_code|escalation_level|resolution_target_hours|resolution_due_date|
+-----------+-------------+----------------+-----------------------+-------------------+
|NULL       |MEDIUM       |3               |48                     |2026-08-26 03:24:26|
+-----------+-------------+----------------+-----------------------+-------------------+



In [0]:
# ============================================================
# MERGE INTO WAREHOUSE.CUSTOMER_MASTER_EXCEPTIONS
# Match on: BusinessDate + CustomerId + ExceptionCategory
# Update only if severity, owner, support team, description,
# or due date has changed.
# ============================================================

target_table = "retailbank_dev.warehouse.customer_master_exceptions"
delta_target = DeltaTable.forName(spark, target_table)

# Safety: if no exceptions to load, skip MERGE to avoid empty-source issues
if exceptions_df.count() > 0:
    
    delta_target.alias("target").merge(
        exceptions_df.alias("source"),
        """
        target.business_date = source.business_date 
        AND target.customer_id = source.customer_id 
        AND target.exception_category = source.exception_category
        """
    ).whenMatchedUpdate(
        condition="""
            COALESCE(target.exception_description, '') <> COALESCE(source.exception_description, '') OR
            COALESCE(target.severity_code, '') <> COALESCE(source.severity_code, '') OR
            COALESCE(target.business_owner, '') <> COALESCE(source.business_owner, '') OR
            COALESCE(target.support_team, '') <> COALESCE(source.support_team, '') OR
            COALESCE(target.resolution_due_date, '1900-01-01') <> COALESCE(source.resolution_due_date, '1900-01-01')
        """,
        set={
            "exception_description": "source.exception_description",
            "severity_code":         "source.severity_code",
            "severity_description":  "source.severity_description",
            "business_area":         "source.business_area",
            "business_owner":        "source.business_owner",
            "support_team":          "source.support_team",
            "escalation_required":   "source.escalation_required",
            "escalation_level":      "source.escalation_level",
            "resolution_due_date":   "source.resolution_due_date",
            "last_updated_date":     "source.assigned_date"
        }
    ).whenNotMatchedInsert(
        values={
            "business_date":           "source.business_date",
            "customer_id":             "source.customer_id",
            "source_system_code":      "source.source_system_code",
            "exception_category":      "source.exception_category",
            "exception_description":   "source.exception_description",
            "severity_code":           "source.severity_code",
            "severity_description":    "source.severity_description",
            "escalation_required":     "source.escalation_required",
            "resolution_target_hours": "source.resolution_target_hours",
            "escalation_level":        "source.escalation_level",
            "business_area":           "source.business_area",
            "business_owner":          "source.business_owner",
            "support_team":            "source.support_team",
            "exception_status":        "source.exception_status",
            "resolution_status":       "source.resolution_status",
            "logged_date":             "source.logged_date",
            "assigned_date":           "source.assigned_date",
            "resolution_due_date":     "source.resolution_due_date",
            "created_date":            "source.assigned_date",
            "last_updated_date":       "source.assigned_date"
        }
    ).execute()

    # Capture MERGE stats from Delta history
    history_df = spark.sql(f"DESCRIBE HISTORY {target_table}")
    latest_merge = history_df.filter("operation = 'MERGE'").orderBy(F.desc("version")).limit(1)
    
    if latest_merge.count() > 0:
        metrics = latest_merge.select("operationMetrics").collect()[0][0]
        rows_inserted = int(metrics.get("numTargetRowsInserted", "0"))
        rows_updated  = int(metrics.get("numTargetRowsUpdated", "0"))
    else:
        rows_inserted = 0
        rows_updated  = 0
        
else:
    rows_inserted = 0
    rows_updated  = 0
    if debug:
        print("No exceptions to MERGE.")

if debug:
    print(f"MERGE complete: {rows_inserted} inserted, {rows_updated} updated")

MERGE complete: 1 inserted, 0 updated


In [0]:
# ============================================================
# AUDIT: EXCEPTION EXECUTION SUMMARY
# Daily snapshot: how many critical/high/medium/low exceptions?
# This feeds management dashboards.
# ============================================================

if exceptions_df.count() > 0:
    summary = exceptions_df.groupBy().agg(
        F.count("*").alias("total_exceptions"),
        F.sum(F.when(F.col("severity_code") == "CRITICAL", 1).otherwise(0)).alias("critical_exceptions"),
        F.sum(F.when(F.col("severity_code") == "HIGH", 1).otherwise(0)).alias("high_exceptions"),
        F.sum(F.when(F.col("severity_code") == "MEDIUM", 1).otherwise(0)).alias("medium_exceptions"),
        F.sum(F.when(F.col("severity_code") == "LOW", 1).otherwise(0)).alias("low_exceptions"),
        F.sum(F.when(F.col("escalation_required") == True, 1).otherwise(0)).alias("escalated_exceptions")
    ).collect()[0]
    
    tot  = summary.total_exceptions
    crit = summary.critical_exceptions or 0
    high = summary.high_exceptions or 0
    med  = summary.medium_exceptions or 0
    low  = summary.low_exceptions or 0
    esc  = summary.escalated_exceptions or 0
else:
    tot = crit = high = med = low = esc = 0

end_time         = datetime.now()
duration_seconds = int((end_time - start_time).total_seconds())

spark.sql(f"""
    INSERT INTO retailbank_dev.audit.exception_execution_summary
    (
        business_date, execution_id, procedure_name, total_exceptions,
        critical_exceptions, high_exceptions, medium_exceptions, low_exceptions,
        escalated_exceptions, execution_seconds, created_date
    )
    VALUES
    (
        '{business_date}', '{execution_id}', '{procedure_name}', {tot},
        {crit}, {high}, {med}, {low}, {esc}, {duration_seconds},
        '{end_time.strftime("%Y-%m-%d %H:%M:%S")}'
    )
""")

if debug:
    print("Exception Execution Summary written to audit.")

Exception Execution Summary written to audit.


In [0]:
# ============================================================
# FINALISE AUDIT
# ============================================================

status  = "SUCCESS"
message = (
    f"Customer Exception Load Completed Successfully. "
    f"Exceptions Read: {rows_read}, "
    f"Exceptions Loaded: {rows_inserted}, "
    f"Duplicate Exceptions Removed: {duplicate_exceptions_removed}"
)

write_audit_end(status, message)

if debug:
    print("=" * 50)
    print("CUSTOMER EXCEPTION LOAD SUMMARY")
    print("=" * 50)
    print(f"Rows Read              : {rows_read}")
    print(f"Exceptions Loaded      : {rows_inserted}")
    print(f"Exceptions Updated     : {rows_updated}")
    print(f"Duplicates Removed     : {duplicate_exceptions_removed}")
    print(f"Status                 : {status}")
    print("=" * 50)
    
    if exceptions_df.count() > 0:
        exceptions_df.groupBy("severity_code").count().orderBy("severity_code").show()

print("nb_02_CustomerMasterExceptions completed successfully.")

CUSTOMER EXCEPTION LOAD SUMMARY
Rows Read              : 11
Exceptions Loaded      : 1
Exceptions Updated     : 0
Duplicates Removed     : 10
Status                 : SUCCESS
+-------------+-----+
|severity_code|count|
+-------------+-----+
|       MEDIUM|    1|
+-------------+-----+

nb_02_CustomerMasterExceptions completed successfully.
